# Build combined k-means input matrices (Quartet multiomics)

Reads the per-modality corrected outputs written by `evaluation_data/multiomics/03_central_RBE.ipynb`
and the uncorrected inputs from `before/`, then stacks the three omics layers into a single
joint matrix per condition (before / central-corrected / FedRBE-corrected).

There is no true shared library ID across the Quartet omics layers. Samples are pseudo-matched
by `pseudo_sample = "{client}_{donor}_{rep}_{i}"` (built in `02_prepare_RBE_inputs.ipynb`).
The three log-scale modality blocks are stacked vertically; downstream `run_central_kmeans`
and the FeatureCloud `fc_kmeans` app each apply their standard per-feature scaling on the
joint matrix (no manual scaling here).

**Reads:**
- `evaluation_data/multiomics/before/<Modality>/central_intensities_log_UNION.tsv` + `metadata.tsv`
- `evaluation_data/multiomics/after/<Modality>/intensities_log_Rcorrected_UNION.tsv` + `metadata.tsv`
- `evaluation_data/multiomics/after/<Modality>/FedApp_corrected_data.tsv` (real federated app output, written by `04_run_fedrbe.ipynb`, optional) — falls back to `FedSim_corrected_data.tsv` (local Python simulation) when FedApp is missing

**Writes:**

Joint matrices used by both the k-means flow and `evaluation/evaluation_multiomics.ipynb` are written into `evaluation_data/` next to the per-modality fedRBE outputs:
- `evaluation_data/multiomics/after/all_modalities_before_kmeans_matrix.tsv`
- `evaluation_data/multiomics/after/all_modalities_corrected_kmeans_matrix.tsv`
- `evaluation_data/multiomics/after/all_modalities_fedsim_kmeans_matrix.tsv` (if FedRBE outputs exist)
- `evaluation_data/multiomics/after/all_modalities_metadata.tsv`

Per-client `design.tsv` / `intensities.tsv` slices are k-means-only inputs and live inside the clustering-eval folder:
- `evaluation_clusterization_after_correction/real_datasets/multiomics/before/<client>/design.tsv`
- `evaluation_clusterization_after_correction/real_datasets/multiomics/before/<client>/intensities.tsv`

**Consumed by:** the standard real-dataset flow in this repo (notebooks `../01_data_preparation.ipynb` … `../04_analysis_metrics_plots.ipynb`).

In [12]:
library(tidyverse)

# Paths are resolved relative to this notebook's location
# (evaluation_clusterization_after_correction/real_datasets/), which is the
# kernel's working directory when the notebook is run from VS Code.
#
# Read paths point at the shared fedRBE data layer in evaluation_data/.
# Per-client k-means splits (writes only) live inside this clustering-eval
# folder so the k-means pipeline is self-contained.
MULTIOMICS_DIR <- file.path("..", "..", "evaluation_data", "multiomics")
BEFORE_DIR <- file.path(MULTIOMICS_DIR, "before")
AFTER_DIR  <- file.path(MULTIOMICS_DIR, "after")
KMEANS_INPUTS_DIR <- file.path("multiomics", "before")

MODALITIES <- c("Transcriptomics", "Proteomics", "Metabolomics")
DONOR_LEVELS <- c("D6", "D5", "F7", "M8")

## Helper functions

In [13]:
read_feature_matrix <- function(path) {
  df <- readr::read_tsv(path, show_col_types = FALSE)
  first_col <- names(df)[1]
  mat <- df %>%
    column_to_rownames(first_col) %>%
    as.data.frame(check.names = FALSE)
  mat[] <- lapply(mat, as.numeric)
  as.matrix(mat)
}

write_feature_matrix <- function(expr, path) {
  out <- as.data.frame(expr, check.names = FALSE) %>% rownames_to_column("rowname")
  write.table(out, file = path, sep = "\t", quote = TRUE, row.names = FALSE, col.names = TRUE)
}

# Reindex columns to pseudo_sample and prefix rownames with the modality so
# they remain unique after vertical concatenation. No manual scaling here:
# central k-means and the FeatureCloud fc_kmeans app each apply per-feature
# scaling on the joint matrix.
build_modality_block <- function(expr, metadata, modality, sample_keys) {
  mat <- expr[, metadata$file, drop = FALSE]
  colnames(mat) <- metadata$pseudo_sample
  if (anyDuplicated(colnames(mat)) > 0) stop(modality, ": duplicate pseudo-sample columns.", call. = FALSE)
  mat <- mat[, sample_keys, drop = FALSE]
  rownames(mat) <- paste0(modality, "__", make.unique(rownames(mat)))
  mat
}

## Load per-modality matrices

In [14]:
before_matrices    <- list()
corrected_matrices <- list()
fedrbe_matrices    <- list()
metadata_by_modality <- list()

for (modality in MODALITIES) {
  before_dir  <- file.path(BEFORE_DIR, modality)
  after_dir   <- file.path(AFTER_DIR,  modality)

  before_matrices[[modality]] <- read_feature_matrix(
    file.path(before_dir, "central_intensities_log_UNION.tsv")
  )
  corrected_matrices[[modality]] <- read_feature_matrix(
    file.path(after_dir, "intensities_log_Rcorrected_UNION.tsv")
  )

  # Prefer the real federated app output; fall back to the local Python
  # simulation (FedSim) if the app was not run. Both files are written by
  # 04_run_fedrbe.ipynb depending on whether Docker / featurecloud are
  # available.
  fedrbe_path <- file.path(after_dir, "FedApp_corrected_data.tsv")
  if (!file.exists(fedrbe_path)) {
    fedrbe_path <- file.path(after_dir, "FedSim_corrected_data.tsv")
  }
  if (file.exists(fedrbe_path)) {
    fedrbe_matrices[[modality]] <- read_feature_matrix(fedrbe_path)
  }

  metadata_by_modality[[modality]] <- readr::read_tsv(
    file.path(after_dir, "metadata.tsv"), show_col_types = FALSE
  ) %>% mutate(
    condition = factor(condition, levels = DONOR_LEVELS),
    batch = factor(batch),
    rep = as.integer(rep)
  )
}

message("Loaded modalities: ", paste(MODALITIES, collapse = ", "))
message("FedRBE outputs available: ", length(fedrbe_matrices) == length(MODALITIES))


Loaded modalities: Transcriptomics, Proteomics, Metabolomics

FedRBE outputs available: TRUE



## Build combined matrices

Verify that all three modalities share the same `pseudo_sample` keys, then stack
the per-modality blocks vertically into joint matrices (one row per
`Modality__feature`, one column per `pseudo_sample`).

In [15]:
sample_keys <- sort(unique(metadata_by_modality[[MODALITIES[1]]]$pseudo_sample))

for (modality in MODALITIES) {
  keys <- sort(unique(metadata_by_modality[[modality]]$pseudo_sample))
  if (!identical(keys, sample_keys)) stop(modality, ": pseudo-sample keys do not match across modalities.", call. = FALSE)
}

before_blocks    <- purrr::map(MODALITIES, ~ build_modality_block(before_matrices[[.x]],    metadata_by_modality[[.x]], .x, sample_keys))
corrected_blocks <- purrr::map(MODALITIES, ~ build_modality_block(corrected_matrices[[.x]], metadata_by_modality[[.x]], .x, sample_keys))

names(before_blocks)    <- MODALITIES
names(corrected_blocks) <- MODALITIES

before_kmeans_matrix    <- do.call(rbind, before_blocks)
corrected_kmeans_matrix <- do.call(rbind, corrected_blocks)

if (length(fedrbe_matrices) == length(MODALITIES)) {
  fedrbe_blocks  <- purrr::map(MODALITIES, ~ build_modality_block(fedrbe_matrices[[.x]], metadata_by_modality[[.x]], .x, sample_keys))
  names(fedrbe_blocks) <- MODALITIES
  fedrbe_kmeans_matrix <- do.call(rbind, fedrbe_blocks)
} else {
  fedrbe_kmeans_matrix <- NULL
  message("Skipping FedRBE matrix — not all modalities have fedrbe outputs.")
}

stopifnot(!anyNA(before_kmeans_matrix), all(is.finite(before_kmeans_matrix)))
stopifnot(!anyNA(corrected_kmeans_matrix), all(is.finite(corrected_kmeans_matrix)))
stopifnot(anyDuplicated(rownames(before_kmeans_matrix)) == 0)
stopifnot(anyDuplicated(rownames(corrected_kmeans_matrix)) == 0)

## Build combined metadata

In [16]:
combined_metadata <- metadata_by_modality[[MODALITIES[1]]] %>%
  select(pseudo_sample, client, condition, rep) %>%
  distinct()

for (modality in MODALITIES) {
  modality_meta <- metadata_by_modality[[modality]] %>%
    select(pseudo_sample, file, batch, lab, platform, protocol, date) %>%
    rename_with(~ paste0(modality, "_", .x), -pseudo_sample)
  combined_metadata <- combined_metadata %>% left_join(modality_meta, by = "pseudo_sample")
}

combined_metadata <- combined_metadata %>%
  mutate(pseudo_sample = factor(pseudo_sample, levels = sample_keys)) %>%
  arrange(pseudo_sample) %>%
  mutate(pseudo_sample = as.character(pseudo_sample))

stopifnot(identical(colnames(before_kmeans_matrix),    combined_metadata$pseudo_sample))
stopifnot(identical(colnames(corrected_kmeans_matrix), combined_metadata$pseudo_sample))
if (!is.null(fedrbe_kmeans_matrix)) {
  stopifnot(identical(colnames(fedrbe_kmeans_matrix), combined_metadata$pseudo_sample))
}

## Write outputs

In [17]:
write_feature_matrix(before_kmeans_matrix,    file.path(AFTER_DIR, "all_modalities_before_kmeans_matrix.tsv"))
write_feature_matrix(corrected_kmeans_matrix, file.path(AFTER_DIR, "all_modalities_corrected_kmeans_matrix.tsv"))
if (!is.null(fedrbe_kmeans_matrix)) {
  write_feature_matrix(fedrbe_kmeans_matrix, file.path(AFTER_DIR, "all_modalities_fedsim_kmeans_matrix.tsv"))
}
write.table(combined_metadata, file = file.path(AFTER_DIR, "all_modalities_metadata.tsv"),
            sep = "\t", quote = TRUE, row.names = FALSE, col.names = TRUE)

tibble(
  matrix = c("before", "corrected", if (!is.null(fedrbe_kmeans_matrix)) "fedrbe" else character(0)),
  features = c(nrow(before_kmeans_matrix), nrow(corrected_kmeans_matrix),
               if (!is.null(fedrbe_kmeans_matrix)) nrow(fedrbe_kmeans_matrix) else integer(0)),
  samples = c(ncol(before_kmeans_matrix), ncol(corrected_kmeans_matrix),
              if (!is.null(fedrbe_kmeans_matrix)) ncol(fedrbe_kmeans_matrix) else integer(0))
)

matrix,features,samples
<chr>,<int>,<int>
before,29517,48
corrected,29517,48
fedrbe,29517,48


## Write per-client `design.tsv` + `intensities.tsv` slices

Drop a flat per-client layout under
`evaluation_clusterization_after_correction/real_datasets/multiomics/before/<client>/`
so the standard `evaluation_clusterization_after_correction/real_datasets/01-04`
notebooks (which use `dataset_configs()` + `discover_clients()`) treat
multiomics like any other dataset. These are k-means-only inputs, so they
live inside the clustering-eval folder rather than in `evaluation_data/`.

Each `<client>/` directory holds a per-client column slice of the joint
**before** k-means matrix (raw log values, with rownames prefixed by
modality) plus a `design.tsv` with `file`, `condition`, and `rep`.

In [18]:
clients <- sort(unique(combined_metadata$client))
message("Per-client splits for ", length(clients), " client(s).")

dir.create(KMEANS_INPUTS_DIR, recursive = TRUE, showWarnings = FALSE)

for (cl in clients) {
  cl_meta <- combined_metadata %>%
    filter(client == cl) %>%
    arrange(pseudo_sample) %>%
    transmute(file = pseudo_sample, condition = condition, rep = rep)

  cl_dir <- file.path(KMEANS_INPUTS_DIR, cl)
  dir.create(cl_dir, recursive = TRUE, showWarnings = FALSE)

  write.table(cl_meta, file = file.path(cl_dir, "design.tsv"),
              sep = "\t", quote = TRUE, row.names = FALSE, col.names = TRUE)

  cl_mat <- before_kmeans_matrix[, cl_meta$file, drop = FALSE]
  out_df <- as.data.frame(cl_mat, check.names = FALSE) %>% rownames_to_column("rowname")
  write.table(out_df, file = file.path(cl_dir, "intensities.tsv"),
              sep = "\t", quote = TRUE, row.names = FALSE, col.names = TRUE)
}

Per-client splits for 3 client(s).



## Session information

In [19]:
sessionInfo()

R version 4.5.2 (2025-10-31)
Platform: x86_64-conda-linux-gnu
Running under: Ubuntu 24.04.4 LTS

Matrix products: default
BLAS/LAPACK: /home/yuliya-cosybio/miniforge3/envs/fedRBE/lib/libopenblasp-r0.3.30.so;  LAPACK version 3.12.0

locale:
 [1] LC_CTYPE=en_US.UTF-8       LC_NUMERIC=C              
 [3] LC_TIME=en_US.UTF-8        LC_COLLATE=en_US.UTF-8    
 [5] LC_MONETARY=en_US.UTF-8    LC_MESSAGES=en_US.UTF-8   
 [7] LC_PAPER=en_US.UTF-8       LC_NAME=C                 
 [9] LC_ADDRESS=C               LC_TELEPHONE=C            
[11] LC_MEASUREMENT=en_US.UTF-8 LC_IDENTIFICATION=C       

time zone: Europe/Berlin
tzcode source: system (glibc)

attached base packages:
[1] stats     graphics  grDevices utils     datasets  methods   base     

other attached packages:
 [1] lubridate_1.9.5 forcats_1.0.1   stringr_1.6.0   dplyr_1.2.1    
 [5] purrr_1.2.1     readr_2.2.0     tidyr_1.3.2     tibble_3.3.1   
 [9] ggplot2_4.0.2   tidyverse_2.0.0

loaded via a namespace (and not attached):
 [1] b